# Segment a scene

Semantic segmentation labels *every point* in a scene. A whole room rarely fits in one forward pass, so this library separates **what** the model computes from **how** it is run over a large scene: that second job belongs to an `Inferer`.

This notebook has two halves:

1. **The inferer contract**, demonstrated on a synthetic room with a toy predictor. This runs anywhere, no model or dataset needed.
2. **A real pretrained model** with its registered preprocessing, evaluated at full point resolution. This part needs a dataset, a sparse-conv extra, and downloaded weights, so it is shown for reference.

New to the library? Start with the [Quickstart](01-quickstart.md).

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import torch

import torch_pointcloud as tp

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## 1. The inferer contract

An [inferer](../inferers/overview.md) takes a packed-batch dict and a `predictor` callable, and returns one prediction per input point, shape $(N, C_\text{out})$, aligned to the input order. The predictor maps a (sub-)scene dict to logits; the inferer decides whether that happens in one pass, over tiled blocks, or under test-time augmentation, and stitches the partial results together.

To see this with no model at all, we build a synthetic $10 \times 10 \times 3$ m room and use a *toy predictor* that labels points by height into four bands. The point is the plumbing, not the labels.

In [ ]:
room = torch.rand(40_000, 3) * torch.tensor([10.0, 10.0, 3.0])
scene = {"pos": room, "batch": torch.zeros(len(room), dtype=torch.long)}


def height_predictor(d):
    """Toy predictor: 4 'classes' by height (z), as confident one-hot logits."""
    z = d["pos"][:, 2]
    band = (z / 3.0 * 4).clamp(0, 3).long()
    return torch.nn.functional.one_hot(band, num_classes=4).float() * 5.0


show_cloud(room, color=height_predictor(scene).argmax(-1), title="ground truth (height bands)");

The simplest inferer just calls the predictor on the whole scene:

In [ ]:
from torch_pointcloud.inferers import SimpleInferer

probs = SimpleInferer()(scene, predictor=height_predictor)
print("output:", tuple(probs.shape), "| aligned to pos:", probs.shape[0] == scene["pos"].shape[0])

When the scene is too large for one pass, `SlidingWindowInferer` tiles it into cubic blocks of a fixed metric size, runs the predictor on each, and blends overlapping predictions. With `mode="gaussian"` points near a block center count more than those at the seam, which softens block boundaries.

The output is still one row per original point, so we can colour the cloud by the stitched prediction exactly as before:

In [ ]:
from torch_pointcloud.inferers import SlidingWindowInferer

inferer = SlidingWindowInferer(block_size=3.0, overlap=0.5, mode="gaussian")
probs = inferer(scene, predictor=height_predictor)

show_cloud(room, color=probs.argmax(-1), title="sliding-window prediction");
print("output:", tuple(probs.shape))

Test-time augmentation wraps *any* base inferer: it runs several augmented passes and averages them, without touching the predictor.

```python
from torch_pointcloud.inferers import TTAInferer
from torch_pointcloud.transforms import Compose, RandomFlip, RandomRotate

tta = TTAInferer(
    base=SlidingWindowInferer(block_size=3.0, overlap=0.5),
    transforms=Compose([
        RandomRotate(keys="pos", angle_range=(-180.0, 180.0), axis=2, p=1.0),
        RandomFlip(keys="pos", axes=[0, 1], p=0.5),
    ]),
    num_passes=4,
)
probs = tta(scene, predictor=height_predictor)
```

`KNNWindowInferer` and `VoxelPartitionInferer` are two more strategies with the same contract. The [Inferers guide](../inferers/overview.md) compares them.

## 2. A real pretrained model

Voxel backbones (SpUNet, SPVCNN, PT-V3) do not consume raw points directly: they voxelize the cloud, run sparse convolutions on the grid, and map predictions back. All of that lives in the **transform pipeline the checkpoint was trained with**, which `create_model(..., return_info=True)` returns alongside the model.

> The cells below need the `spconv` extra, the ScanNet dataset, and the downloaded checkpoint. They mirror `examples/spunet_benchmark_scannet.py`; the synthetic demo above needs none of that.

In [ ]:
model, info = tp.create_model(
    "spunet-v1m1.scannet20.pointcept", 
    task="segmentation", 
    pretrained=True, 
    return_info=True,
)
model = model.eval().to(device)
transform = info["transform"]  # the exact preprocessing this checkpoint expects

The transform voxelizes the scene and writes the model inputs into the dict: features `x`, grid coordinates `pos`, and an `inverse` map from each raw point to its voxel. `collate` then adds the packed `batch` index.

In [ ]:
from torch_pointcloud.datasets import ScanNet20
from torch_pointcloud.utils.data import collate

scene = ScanNet20(root="data", split="val", transform=transform)[0]
batch = collate([scene])
print({k: tuple(v.shape) for k, v in batch.items() if torch.is_tensor(v)})

Run the model once on the whole voxelized scene, then broadcast the voxel logits back to the raw points through the `inverse` map. This evaluates at full point resolution, which is how the reported mIoU is measured.

In [ ]:
with torch.no_grad():
    voxel_logits = model(batch["x"].to(device), batch["pos"].to(device), batch["batch"].to(device))

preds = voxel_logits[batch["inverse"]].argmax(dim=-1).cpu()  # (N_raw,) one label per raw point
print("raw points:", preds.shape[0])

In [ ]:
from torch_pointcloud.utils.metrics import confusion_matrix

# Metric at full (raw) resolution: voxel predictions broadcast back through `inverse`.
target = batch["origin_segment"]  # raw-resolution ground truth (0..19, -1 = ignore)
cm = confusion_matrix(preds, target, model.num_classes, ignore_index=-1)
iou = cm.diag() / (cm.sum(0) + cm.sum(1) - cm.diag()).clamp_min(1)
print(f"scene mIoU: {iou.mean():.3f}")

# Visualize at voxel resolution: the transform voxelizes the room in place, so `pos`
# and the per-voxel `segment` are the model's view of the scene.
voxel_pred = voxel_logits.argmax(-1).cpu()
fig = plt.figure(figsize=(8, 4))
show_cloud(batch["pos"], color=batch["segment"], ax=fig.add_subplot(121, projection="3d"), title="ground truth", cmap="tab20")
show_cloud(batch["pos"], color=voxel_pred, ax=fig.add_subplot(122, projection="3d"), title="prediction", cmap="tab20");

## Recap

- An inferer maps `(scene dict, predictor) -> per-point logits`, aligned to the input.
- For scenes that fit, run the model once and (for voxel models) broadcast back through `inverse`.
- For scenes that do not fit, tile with `SlidingWindowInferer`, optionally wrapped in `TTAInferer`.

Next: build the preprocessing pipelines a checkpoint expects in [Preprocessing pipelines](03-transforms.md).